# NoRA vs LoRA on coffee-order parsing

Compares **LoRA**, **NoRA-init** and **NoRA** when fine-tuning Qwen3.5-0.8B to parse noisy
Russian/English order transcripts into the compact format (`cap L oat;lat x2`, `?` = unclear).

Method reference: Jiale Kang, Ziyin Yue, Zheng Zhan, Yangyi Huang, Weiyang Liu,
*Normalized Low-Rank Adaptation*, arXiv:2608.31036 (2026). NoRA constrains each column of
`lora_A.weight` (shape `r x in_features`) to unit L2 norm along the rank dimension.

- **NoRA-init** — normalize once after initialization, then train as plain LoRA.
- **NoRA** — keep the constraint throughout training.

Implementation: the official PEFT fork (github.com/Joluck/NoRA, `use_nora=True` / `use_nora="init"`),
verified compatible with transformers 5. `coffee_parser.nora` carries an independent
`torch.nn.utils.parametrize` reimplementation that the test suite cross-checks against the fork
and against the closed form `dW = alpha/r * B @ Norm(A)`.

No Unsloth anywhere: transformers + TRL + PEFT for every method, identical stack and settings
except the `LoraConfig`.

**Deliberate deviations from the paper's defaults are listed in the last cell. Read it before
quoting any number.**

## 1. Environment

Install the NoRA fork **last** — anything that resolves `peft` from PyPI afterwards will
silently replace it, and the assert below is what catches that.

In [ ]:
INSTALL = True   # set False on a server where the env is already built

if INSTALL:
    %pip install -q "transformers>=5" trl datasets matplotlib pandas
    %pip install -q git+https://github.com/Joluck/NoRA.git#subdirectory=peft

from peft import LoraConfig
assert "use_nora" in LoraConfig.__dataclass_fields__, (
    "NoRA fork missing or overwritten by upstream peft; reinstall it last")
print("NoRA fork present")

In [ ]:
import os, sys, json

# The repo provides the generator, menu, parse_compact and the hand-written set.
# Already on the server; cloned in Colab.
REPO = os.environ.get("COFFEE_LLM_REPO", "/content/coffee-llm")
if not os.path.isdir(os.path.join(REPO, "src")):
    for guess in ("..", ".", "/content/coffee-llm"):
        if os.path.isdir(os.path.join(guess, "src/coffee_parser")):
            REPO = os.path.abspath(guess)
            break
    else:
        raise SystemExit("Set COFFEE_LLM_REPO or clone the repo first")

sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)

from coffee_parser import experiment as X
from coffee_parser import generator as gen

ENV = X.describe_env()
TOK = X.tokenizer()
for k, v in ENV.items():
    print(f"{k:14} {v}")

if not ENV["bf16"]:
    print("\nWARNING: no bf16 -> running in fp32 (Qwen3.5 is unstable in fp16 on a T4).")
    print("Expect roughly 5x the wall time below. A T4 is not viable for the full grid.")

## 2. Configuration

Everything tunable lives here. The default is the **pilot**: rank 16, sizes 100/300/1000,
3 seeds, 4 methods.

`lora_alpha_r` is LoRA with `alpha = r` — the paper's LoRA baseline uses `alpha = 2r` while NoRA
uses `alpha = r`, so without this arm a NoRA win is confounded with the scaling change.

In [ ]:
GRID = X.Grid(
    methods=("lora", "lora_alpha_r", "nora_init", "nora"),
    sizes=(100, 300, 1000),
    ranks=(16,),
    seeds=(0, 1, 2),
    max_steps=250,
    batch_size=16,
    warmup_steps=10,
    eval_every=25,
    lr_candidates=(1e-4, 2e-4, 5e-4),
)

# Full grid for the paper (run after the pilot looks sane):
# GRID = X.Grid(sizes=(100, 300, 1000, 3800), ranks=(4, 16), seeds=(0, 1, 2))

print(f"{len(GRID.methods) * len(GRID.sizes) * len(GRID.ranks) * len(GRID.seeds)} grid runs"
      f" + {len(GRID.methods) * len(GRID.lr_candidates)} LR-sweep runs")

In [ ]:
# Drive in Colab so a disconnect does not lose results; a local dir otherwise.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/coffee-nora"
except Exception:
    OUT = os.path.join(REPO, "runs")

for sub in ("", "curves", "adapters", "figures", "tables"):
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
RESULTS = os.path.join(OUT, "results.jsonl")
print("writing to", OUT)
print(len(X.load_results(RESULTS)), "results already logged (these will be skipped)")

## 3. Data

Splits come from `coffee_parser.generator`, seeded per split and de-duplicated by input string
against every earlier split. The `hard` split raises item counts, typos, corrections and noise:
the production eval sits at a 98% ceiling, which cannot separate the methods.

The 49 hand-written cases from `testset.py` are reported separately — at n=49 a single example
moves the score by 2 points, so it is an anecdote, not a measurement.

In [ ]:
gen.assert_menu_consistent()
SPLITS = X.build_splits(train_pool=3800, val_n=300, test_n=1000, hard_n=1000)

all_inputs = [(name, {e["input"] for e in s}) for name, s in SPLITS.items()]
for i, (na, a) in enumerate(all_inputs):
    for nb, b in all_inputs[i + 1:]:
        assert not a & b, f"{na} and {nb} overlap"

for name, s in SPLITS.items():
    items = sum(len(e["output"].split(";")) for e in s) / len(s)
    refusals = sum(e["output"].strip() == "?" for e in s) / len(s)
    print(f"{name:6} n={len(s):5}  mean items={items:.2f}  pure refusals={refusals:.0%}")

print()
for e in SPLITS["hard"][:5]:
    print(f"  {e['input']}\n    -> {e['output']}")

## 4. Cost estimate

Time one short run before committing to the grid.

In [ ]:
import copy, time

probe = copy.copy(GRID)
probe.max_steps = 20
probe.eval_every = 10
t0 = time.perf_counter()
_ = X.train_one("lora", GRID.ranks[0], 300, 0, 2e-4, SPLITS, probe, ENV, TOK, OUT)
probe_s = time.perf_counter() - t0

per_run = probe_s * (GRID.max_steps / probe.max_steps) * 0.6 + probe_s * 0.4
n_runs = len(GRID.methods) * len(GRID.sizes) * len(GRID.ranks) * len(GRID.seeds)
n_sweep = len(GRID.methods) * len(GRID.lr_candidates)
print(f"20-step probe: {probe_s/60:.1f} min")
print(f"estimated per full run: {per_run/60:.1f} min")
print(f"LR sweep ({n_sweep} runs): {n_sweep*per_run/3600:.1f} h")
print(f"grid ({n_runs} runs): {n_runs*per_run/3600:.1f} h")

## 5. Learning-rate selection

One LR per method, chosen on **val** exact match at size 1000 / rank 16 / seed 0, then frozen for
the whole grid. The test splits are never consulted here.

In [ ]:
GRID.lrs = X.select_lrs(GRID, SPLITS, ENV, TOK, OUT, RESULTS)
print(GRID.lrs)

## 6. Grid

Resumable: runs already in `results.jsonl` are skipped, so re-running this cell after a
disconnect continues where it stopped.

In [ ]:
X.run_grid(GRID, SPLITS, ENV, TOK, OUT, RESULTS, dry_run=True)

In [ ]:
X.run_grid(GRID, SPLITS, ENV, TOK, OUT, RESULTS)

## 7. Results

Sanity checks first: every run must have passed its invariants, or its numbers mean nothing.

In [ ]:
import pandas as pd

ROWS = [r for r in X.load_results(RESULTS) if r.get("phase") == "grid"]
print(len(ROWS), "grid runs")

bad = [r["run_id"] for r in ROWS
       if not r["merge_equivalent"]
       or not r["invariants_start"]["lora_B_zero"]
       or not r["invariants_start"]["delta_weight_matches_formula"]]
print("runs failing invariants:", bad or "none")

impl = {r["implementation"] for r in ROWS}
assert len(impl) <= 1, f"mixed implementations in one results file: {impl}"
print("implementation:", impl)

# NoRA keeps the constraint to the end; NoRA-init does not have to.
for r in ROWS:
    if r["method"] == "nora":
        e = r["invariants_end"]["effective_colnorm"]
        assert abs(e["max"] - 1) < 1e-4 and abs(e["min"] - 1) < 1e-4, r["run_id"]
print("NoRA column norms still unit at end of training: ok")

In [ ]:
def table(split):
    df = pd.DataFrame(X.summarize(ROWS, split))
    if df.empty:
        return df
    df["mean_pm_std"] = df.apply(lambda r: f"{r['mean']:.3f} +/- {r['std']:.3f}", axis=1)
    df["ci95"] = df.apply(
        lambda r: "-" if r["ci_lo"] is None else f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]", axis=1)
    return df[["method", "rank", "size", "n_seeds", "mean_pm_std", "ci95"]]

for split in ["hard", "test", "val", "hand"]:
    print(f"\n=== exact match, {split} split ===")
    print(table(split).to_string(index=False))

In [ ]:
aux = pd.DataFrame([
    {"method": r["method"], "size": r["size"], "rank": r["rank"], "seed": r["seed"],
     "parse_validity": r["metrics"]["hard"]["parse_validity"],
     "refusal_P": r["metrics"]["hard"]["refusal_precision"],
     "refusal_R": r["metrics"]["hard"]["refusal_recall"],
     "steps_to_90": r["steps_to_90_val_em"]}
    for r in ROWS])
print(aux.groupby(["method", "size"]).mean(numeric_only=True).round(3).to_string())

In [ ]:
# Paired by (rank, size, seed): the only comparison that cancels seed noise.
pairs = []
for a, b in [("nora_init", "lora"), ("nora", "lora"),
             ("nora_init", "lora_alpha_r"), ("nora", "lora_alpha_r"), ("nora", "nora_init")]:
    pairs += X.paired(ROWS, "hard", a, b)
pdf = pd.DataFrame(pairs)
if not pdf.empty:
    pdf["verdict"] = pdf["within_noise"].map({True: "within noise", False: "separated"})
    print(pdf[["a", "b", "rank", "size", "n_pairs", "mean_diff", "cohens_dz",
               "ci_lo", "ci_hi", "verdict"]].round(3).to_string(index=False))
    print("\nA 95% CI straddling 0 means this experiment does not separate the methods.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"lora": "#4e79a7", "lora_alpha_r": "#8fb3d9", "nora_init": "#f28e2b", "nora": "#e15759"}


def fig_em_vs_size(split="hard", rank=None):
    rank = rank or GRID.ranks[0]
    df = pd.DataFrame(X.summarize(ROWS, split))
    df = df[df["rank"] == rank]
    fig, ax = plt.subplots(figsize=(5, 3.4))
    for method, g in df.groupby("method"):
        g = g.sort_values("size")
        ax.plot(g["size"], g["mean"], marker="o", label=method, color=COLORS.get(method))
        if g["ci_lo"].notna().all():
            ax.fill_between(g["size"], g["ci_lo"], g["ci_hi"], alpha=0.15, color=COLORS.get(method))
    ax.set_xscale("log")
    ax.set_xticks(sorted(df["size"].unique()))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel("training examples")
    ax.set_ylabel(f"exact match ({split})")
    ax.set_title(f"rank {rank}, mean over {df['n_seeds'].max()} seeds, 95% bootstrap CI")
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig


def load_curves(**sel):
    out = []
    for r in ROWS:
        if all(r[k] == v for k, v in sel.items()):
            p = os.path.join(OUT, "curves", f"{r['run_id']}.json")
            if os.path.exists(p):
                out.append((r, json.load(open(p))))
    return out


def fig_curves(key, size, rank=None, limit=100, ylabel=None):
    rank = rank or GRID.ranks[0]
    fig, ax = plt.subplots(figsize=(5, 3.4))
    for method in GRID.methods:
        series = [c[key] for _, c in load_curves(method=method, size=size, rank=rank)]
        series = [np.array([p for p in s if p[0] <= limit]) for s in series if s]
        if not series:
            continue
        steps = series[0][:, 0]
        vals = np.stack([s[:, 1] for s in series if len(s) == len(steps)])
        mean = vals.mean(0)
        ax.plot(steps, mean, label=method, color=COLORS.get(method))
        if len(vals) > 1:
            sd = vals.std(0)
            ax.fill_between(steps, mean - sd, mean + sd, alpha=0.15, color=COLORS.get(method))
    ax.set_xlabel("optimizer step")
    ax.set_ylabel(ylabel or key)
    ax.set_title(f"n={size}, rank {rank}")
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig


FIGS = {}
FIGS["em_vs_size_hard"] = fig_em_vs_size("hard")
FIGS["em_vs_size_test"] = fig_em_vs_size("test")
mid = GRID.sizes[min(2, len(GRID.sizes) - 1)]
FIGS["loss_first_100"] = fig_curves("loss", mid, limit=100, ylabel="training loss")
FIGS["gradnorm_first_50"] = fig_curves("grad_norm", mid, limit=50, ylabel="gradient norm")
FIGS["val_em_vs_step"] = fig_curves("val_em", mid, limit=GRID.max_steps, ylabel="val exact match")
plt.show()

In [ ]:
# CSV + LaTeX tables and PDF figures, for the paper.
for split in ["hard", "test", "val", "hand"]:
    df = pd.DataFrame(X.summarize(ROWS, split))
    if df.empty:
        continue
    df.to_csv(os.path.join(OUT, "tables", f"em_{split}.csv"), index=False)
    with open(os.path.join(OUT, "tables", f"em_{split}.tex"), "w") as f:
        f.write(df.round(4).to_latex(index=False,
                caption=f"Exact match on the {split} split, mean over seeds with 95\\% bootstrap CI.",
                label=f"tab:em-{split}"))

pdf_pairs = pd.DataFrame(pairs)
if not pdf_pairs.empty:
    pdf_pairs.to_csv(os.path.join(OUT, "tables", "paired.csv"), index=False)
    with open(os.path.join(OUT, "tables", "paired.tex"), "w") as f:
        f.write(pdf_pairs.round(4).to_latex(index=False,
                caption="Paired per-seed differences in exact match on the hard split.",
                label="tab:paired"))

for name, fig in FIGS.items():
    fig.savefig(os.path.join(OUT, "figures", f"{name}.pdf"), bbox_inches="tight")

print("wrote tables and figures to", OUT)
print(sorted(os.listdir(os.path.join(OUT, "tables"))))
print(sorted(os.listdir(os.path.join(OUT, "figures"))))

## 8. Caveats to carry into the paper

1. **Scaling is confounded with normalization in the paper's own baseline.** LoRA uses
   `alpha = 2r`, NoRA `alpha = r`. The `lora_alpha_r` arm exists to separate the two; quote it
   alongside any NoRA-vs-LoRA claim.
2. **Continuous NoRA does not constrain the stored parameter.** The fork normalizes inside
   `forward()` and `get_delta_weight()`, so `lora_A.weight` itself drifts. Invariant checks test
   the effective matrix `A/||A||`; a check on the raw parameter would fail against the reference
   implementation.
3. **The epsilon is additive**, matching the fork: `A / (||A|| + 1e-6)`. In fp32, column norms
   sit at `1 - O(1e-5)`, never exactly 1.
4. **Under bf16, NoRA-init is only approximately normalized.** The fork normalizes `lora_A` and
   then calls `_move_adapter_to_device_of_base_layer`, which round-trips the weights through the
   base dtype, leaving columns unit-norm to about `2e-3` while `lora_A.dtype` still reports fp32.
   That rounding is ~1000x the epsilon, so on GPU it is the only term that matters. `norm_atol()`
   derives the tolerance from the base dtype as well as the adapter dtype.
5. **BIMI is not implemented in the published fork.** `bimi_init()` is `with torch.no_grad(): pass`
   with the block-identity code commented out, so `init_lora_weights="bimi"` silently gives
   default LoRA init. There is no BIMI arm here for that reason.
6. **The eval data and the training data share one generator**, so all splits inherit the same
   author's assumptions about how people order coffee. The `hard` split raises difficulty but not
   realism. Real speech-to-text transcripts are needed before any of these numbers describes
   production behaviour.
7. **Fixed 250-step budget** means small sizes see many epochs (n=100 at batch 16 is ~40 passes),
   so low-data columns measure memorization as much as learning. That is the intended control
   for optimizer steps, but it is not the same experiment as training to convergence.
8. **The hand-written set is n=49.** One example is worth 2 points. Report it as an anecdote.
9. **Curves use a 150-example val subset** so that eval does not dominate wall time; the logged
   `val` metric uses all 300.